### Text Splitters in LangChain

Text splitters are essential utilities in LLM applications used to break large documents into smaller, semantically meaningful chunks (e.g., paragraphs or sentences). This ensures text fits within model context windows and improves retrieval precision.

#### Why Do We Need Text Splitters?
* **Context Window Limits:** LLMs have token limits; passing an entire multi-page document or book in a single prompt will fail or truncate.
* **Granular Retrieval:** Smaller chunks allow vector databases to locate precise passages rather than broad, unfocused sections.
* **Overlap Management:** Splitters allow overlapping text between consecutive chunks so context isn't lost mid-sentence.


### 1. Length Based Text Splitting

It splits the text into fixed number of characters or tokens


In [7]:
from langchain_text_splitters import CharacterTextSplitter

text ="""
Life is an intricate tapestry woven from a series of fleeting moments, quiet choices, and profound transitions. From the earliest days of human existence, individuals have grappled with its fundamental meaning, seeking to understand a journey that is equal parts fragile and resilient. It is not defined merely by the span of years between birth and death, but by the depth of experiences, the lessons learned through adversity, and the connections forged with others along the way.

At its core, life is dynamic and ever-changing. Change is the only constant, reminding us that no season—whether marked by intense joy or profound sorrow—lasts forever. This impermanence gives life its poignant beauty. When we accept that moments are transient, we learn to appreciate the present rather than constantly looking toward an uncertain future or dwelling on the past. Every morning brings a fresh horizon, offering an opportunity to reset, reinvent, and choose a path aligned with our deepest values.

Growth rarely happens in comfort; instead, it is forged in the crucible of challenges. Obstacles, failures, and heartbreaks are not interruptions to life—they are life itself unfolding. They test our endurance, build our character, and teach us empathy. Without experiencing hardship, our capacity for joy would remain muted, for contrast is required to truly understand light. Overcoming difficulties allows us to discover inner strengths we never knew we possessed, transforming vulnerability into quiet resilience.

Equally important are the relationships that populate our days. Human beings are inherently wired for connection, and the bonds we share with family, friends, and community form the bedrock of a meaningful existence. Sharing laughter, offering support during difficult times, and listening to the stories of others expand our worldview. In giving of ourselves, we often find our own purpose, realizing that a life well-lived is measured not by individual accolades, but by the positive impact we leave on those around us.

Ultimately, life is a blank canvas awaiting our unique brushstrokes. It invites us to be curious, to pursue our passions with courage, and to live authentically. While we cannot control all the external circumstances that come our way, we retain absolute authority over how we respond to them. Embracing this agency allows us to craft a journey rich in purpose, wonder, and deep gratitude for the gift of existence.
"""

splitter = CharacterTextSplitter(
    separator=' ',
    chunk_size=50,  # Maximum number of characters per chunk
    chunk_overlap=20 # Number of characters to overlap between consecutive chunks
)
result = splitter.split_text(text)
print(len(result))
print(result)

82
['Life is an intricate tapestry woven from a series', 'woven from a series of fleeting moments, quiet', 'moments, quiet choices, and profound transitions.', 'transitions. From the earliest days of human', 'days of human existence, individuals have grappled', 'have grappled with its fundamental meaning,', 'fundamental meaning, seeking to understand a', 'to understand a journey that is equal parts', 'that is equal parts fragile and resilient. It is', 'and resilient. It is not defined merely by the', 'merely by the span of years between birth and', 'between birth and death, but by the depth of', 'but by the depth of experiences, the lessons', 'the lessons learned through adversity, and the', 'adversity, and the connections forged with others', 'forged with others along the way.\n\nAt its core,', 'way.\n\nAt its core, life is dynamic and', 'life is dynamic and ever-changing. Change is the', 'Change is the only constant, reminding us that no', 'reminding us that no season—whether marked 

### 2. Recursive Character Text Splitter (text-structured based splitting)
 
It attempts to split text sequentially using a prioritized list of separators—starting with double newlines(\n\n - for paragraphs), single newlines(\n - for new lines), spaces(" " - for word seperation), and characters—to keep paragraphs, sentences, and words intact.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 40,
    chunk_overlap =0,
)

chunks = splitter.split_text(text)

print(len(chunks))

print(chunks)

70
['Life is an intricate tapestry woven', 'from a series of fleeting moments,', 'quiet choices, and profound', 'transitions. From the earliest days of', 'human existence, individuals have', 'grappled with its fundamental meaning,', 'seeking to understand a journey that is', 'equal parts fragile and resilient. It', 'is not defined merely by the span of', 'years between birth and death, but by', 'the depth of experiences, the lessons', 'learned through adversity, and the', 'connections forged with others along', 'the way.', 'At its core, life is dynamic and', 'ever-changing. Change is the only', 'constant, reminding us that no', 'season—whether marked by intense joy or', 'profound sorrow—lasts forever. This', 'impermanence gives life its poignant', 'beauty. When we accept that moments are', 'transient, we learn to appreciate the', 'present rather than constantly looking', 'toward an uncertain future or dwelling', 'on the past. Every morning brings a', 'fresh horizon, offering an opportu

### 3. Document-Structured Based

This is similar to Recursivecharacter Splitting but for documents based on their seperators

This can be used for files like Markdown,Python  as they follow different semantics compared to other

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,Language

text ="""
class CricketPlayer:

    def __init__(self, name, role):
        self.name = name
        self.role = role

    def show_info(self):
        print(f"Player: {self.name} | Role: {self.role}")


# Create an object of the class
player1 = CricketPlayer("Virat Kohli", "Batsman")
player1.show_info()
"""

splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size = 200,
    chunk_overlap=0,
)

result = splitter.split_text(text)

print(len(result))
print(result[0])

2
class CricketPlayer:

    def __init__(self, name, role):
        self.name = name
        self.role = role

    def show_info(self):
        print(f"Player: {self.name} | Role: {self.role}")


### 4. Semantic Meaning Based 

Semantic-based text splitters break documents apart based on meaning rather than fixed character counts. They analyze embedding vectors of sentences to detect shifts in context or topic. Chunks are then created wherever a significant semantic drop or boundary occurs between adjacent segments. This ensures related concepts stay together in the same chunk.

This is done using the sliding window and a split is formed if there is a abrupt difference in the threshold

- This type of chunking is still in developing stage and not really useful in real-world scenarios

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

# Initialize the embedding model
embeddings = OpenAIEmbeddings()

# Create the semantic chunker
text_splitter = SemanticChunker(embeddings)

# Sample document with a shift in topic
text = (
    "Cricket is a popular bat-and-ball team sport played globally. "
    "The main formats include Test matches, ODIs, and T20s. "
    "Meanwhile, Python is a versatile programming language used for machine learning. "
    "Data scientists use libraries like Pandas and NumPy for analysis."
)

# Split text based on meaning
docs = text_splitter.create_documents([text])

for i, doc in enumerate(docs):
    print(f"Chunk {i+1}: {doc.page_content}\n")